[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/32_topk_sampling.ipynb)

# 🟠 Medium: Top-k / Top-p (Nucleus) Sampling

Implement **sampling with top-k and top-p filtering** — the standard LLM decoding strategy.

### Signature
```python
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0) -> int:
    # logits: (V,) unnormalized log-probabilities
    # Returns: sampled token index
```

### Algorithm
1. Scale by temperature: `logits /= temperature`
2. Top-k: keep only top-k logits, set rest to `-inf`
3. Top-p: sort by prob, mask tokens where cumulative prob exceeds p
4. Sample from filtered distribution

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE
def sample_top_k_top_p(logits, top_k=0, top_p=1.0, temperature=1.0):
    # temperature, top-k filter, top-p filter, sample

    # apply temperature
    logits = logits / temperature
    if top_k > 0:
        top_k_values, top_k_indices = torch.topk(logits, k = top_k, dim = -1)
        mask = torch.ones_like(logits, dtype = torch.bool)
        mask[top_k_indices] = False
        logits[mask] = float('-inf')
    if top_p > 0:
        prob = torch.softmax(logits, dim = -1)
        sorted_prob, sorted_indices = torch.sort(prob, dim=-1, descending=True)
        # Top-p (nucleus sampling) works on the cumulative sum of sorted probabilities
        cumulative_prob = torch.cumsum(sorted_prob, dim=-1)
        mask = cumulative_prob - sorted_prob >= top_p  # tokens beyond the nucleus
        sorted_prob[mask] = 0.0
        sorted_prob /= sorted_prob.sum()  # dont forget to renormalize
        sampled_idx = torch.multinomial(sorted_prob, 1)
        return sorted_indices[sampled_idx].item()

    prob = torch.softmax(logits, dim=-1)
    return torch.multinomial(prob, 1).item()

In [7]:
# 🧪 Debug
logits = torch.tensor([1.0, 5.0, 2.0, 0.5])
print('top_k=1:', sample_top_k_top_p(logits.clone(), top_k=1))
print('top_p=0.5:', sample_top_k_top_p(logits.clone(), top_p=0.5))
print('temp=0.01:', sample_top_k_top_p(logits.clone(), temperature=0.01))

top_k=1: 1
top_p=0.5: 1
temp=0.01: 1


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('topk_sampling')


🧪 Testing: Top-k / Top-p Sampling (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] top_k=1 always returns argmax (3.2ms)
  ✅ [2/4] Low temperature concentrates (7.7ms)
  ✅ [3/4] All tokens reachable (no filtering) (120.3ms)
  ✅ [4/4] Returns valid index (2.3ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (133.5ms total)
  Progress saved. Run status() to see your dashboard.

